# Step 3 — Clean Hourly Bike Count Data (5-min source only)

**Input**: `output/hamburg_bike_counts_hourly.csv` (output of `2_Bike_hourly_aggregation.ipynb`)
**Output**: `output/hamburg_bike_counts_hourly_5min_clean_v2.csv.gz`

## Cleaning decisions made in this step (read this first before handover)

1. **Keep only `resolution == '5-min'`** (`HH_STA_HamburgerRadzaehlnetz`), drop `daily` and
   `unknown`. Reason: the `daily` service (`HH_STA_Verkehrsdaten_Rad_Infrarotdetektoren`) actually
   stores a **whole day's total** crammed into a single hourly row (fixed at 22:00 or 23:00 UTC,
   `readings_expected=1`) -- it is not a real hourly value. Mixing it into hourly analysis would
   badly distort hour-level statistics.
2. **Analysis period filter**: 2025-01-01 00:00 to 2026-02-28 23:00 (inclusive). The raw file
   actually extends to 2026-04-25; anything beyond the period is dropped.
3. **Stations named "veraltet" are treated as normal stations, not excluded.** We found that all
   328 of the 5-min stations happen to have "(veraltet)" in their name -- excluding them literally
   would wipe out the entire 5-min data source, so they are kept.
4. **Timezone: no conversion is applied.** The tz label on `hour_utc` is stripped (not
   `tz_convert`) and the numbers are used as-is as Hamburg local time.
   Known risk: testing `hour_utc`'s behavior around DST transitions (the `daily` rows are pinned
   to 22:00/23:00 UTC depending on season; the 5-min rows show no missing/duplicate hour around DST
   transition dates) strongly suggests this column is actually true UTC, not local time. The data
   owner explicitly decided "do not convert, treat as local time already." If this needs to be
   reversed later with `tz_convert('Europe/Berlin')`, only this one step needs to change.
5. **low_coverage (205 rows, 0.006%) kept, not dropped**, only flagged -- the share is extremely
   small and spread across 63 stations and many different dates, no systemic issue.
6. **Extreme values are flagged, not dropped**, using two thresholds side by side:
   - `extreme_flag_relative_p995`: each station's own 99.5th percentile, catching spikes that are
     unusual *relative to that station itself*.
     - Note: the original approach used Tukey's IQR method (`Q3+3*IQR`), but 21 of the 328
       stations have Q1=Q3=0 (chronically low traffic), which collapses the IQR to 0 and flags any
       nonzero value as "extreme" (this mis-flagged 43,388 rows, of which 15,371 were simply
       count=1). Switching to the percentile method fixed this: flagged rows dropped to 14,457
       (0.448%), and the minimum flagged value became 2.
   - `extreme_count_gt1000` / `extreme_count_gt2000`: fixed absolute thresholds shared across all
     stations, useful for "citywide comparison" scenarios (only about 11 major-corridor stations
     ever get flagged).
7. `quality_flag`, a 3-tier version: `ok` (coverage_pct=100) / `partial_coverage` (50-99.9) /
   `low_coverage` (<50) -- consistent with the `low_coverage` boolean, just with one extra tier.


In [1]:
import pandas as pd
import numpy as np

INPUT_PATH = "output/hamburg_bike_counts_hourly.csv"
OUTPUT_PATH = "output/hamburg_bike_counts_hourly_5min_clean_v2.csv.gz"

ANALYSIS_START = pd.Timestamp('2025-01-01 00:00:00', tz='UTC')
ANALYSIS_END = pd.Timestamp('2026-02-28 23:59:59', tz='UTC')

In [2]:
# 1. Load the raw data
df = pd.read_csv(
    INPUT_PATH,
    dtype={
        'station_name': 'category', 'datastream_id': 'int64', 'station_id': 'int64',
        'service_name': 'category', 'layer_name': 'category', 'resolution': 'category',
        'longitude_wgs84': 'float64', 'latitude_wgs84': 'float64',
        'bike_count_hourly': 'float64', 'readings_in_hour': 'int64', 'readings_expected': 'int64',
        'coverage_pct': 'float64', 'low_coverage': 'bool',
    },
    parse_dates=['hour_utc'],
)
print(f"Raw row count: {len(df):,}")

Raw row count: 3,380,782


In [3]:
# 2. Keep only 5-min resolution, drop daily / unknown
df = df[df['resolution'] == '5-min'].copy()

# 3. Filter to the analysis period
df = df[(df['hour_utc'] >= ANALYSIS_START) & (df['hour_utc'] <= ANALYSIS_END)].copy()
print(f"Rows after cleaning + period filter: {len(df):,}  Stations: {df['station_id'].nunique()}")

# veraltet-named stations are intentionally NOT filtered out (see note #3 above)

Rows after cleaning + period filter: 3,227,633  Stations: 328


In [4]:
# 4. Timezone handling: keep the original hour_utc string, add a clean local-time column
#    (no timezone conversion -- see note #4 above)
df['datetime_hamburg'] = df['hour_utc'].dt.tz_localize(None)
df['date'] = df['datetime_hamburg'].dt.date
df['year'] = df['datetime_hamburg'].dt.year
df['month'] = df['datetime_hamburg'].dt.strftime('%Y-%m')
df['hour_24'] = df['datetime_hamburg'].dt.strftime('%H')

# hour_utc keeps its original string format (YYYY-MM-DD HH:MM:SS+00:00), values unchanged
df['hour_utc'] = df['hour_utc'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
df['hour_utc'] = df['hour_utc'].str.replace(r'(\d{2})(\d{2})$', r'\1:\2', regex=True)

In [5]:
# 5. quality_flag, 3-tier
def qflag(cov):
    if cov >= 100:
        return 'ok'
    elif cov >= 50:
        return 'partial_coverage'
    else:
        return 'low_coverage'

df['quality_flag'] = df['coverage_pct'].apply(qflag)
print(df['quality_flag'].value_counts())

quality_flag
ok                  3227075
partial_coverage        353
low_coverage            205
Name: count, dtype: int64


In [6]:
# 6. Extreme-value flags (not dropped -- see note #6 above)
grp = df.groupby('station_id', observed=True)['bike_count_hourly']

# 6a. Relative threshold: each station's own 99.5th percentile (replaces the flawed Tukey IQR method)
p995 = grp.transform(lambda x: x.quantile(0.995))
df['extreme_flag_relative_p995'] = df['bike_count_hourly'] > p995

# 6b. Absolute threshold: fixed values shared network-wide, for cross-station comparison
df['extreme_count_gt1000'] = df['bike_count_hourly'] > 1000
df['extreme_count_gt2000'] = df['bike_count_hourly'] > 2000

print("extreme_flag_relative_p995:", df['extreme_flag_relative_p995'].sum(),
      f"({df['extreme_flag_relative_p995'].mean()*100:.3f}%)")
print("extreme_count_gt1000:", df['extreme_count_gt1000'].sum())
print("extreme_count_gt2000:", df['extreme_count_gt2000'].sum())

extreme_flag_relative_p995: 14457 (0.448%)
extreme_count_gt1000: 82
extreme_count_gt2000: 3


In [7]:
# 7. Reorder columns and write out (gzip-compressed; raw CSV is ~630MB, compressed ~20-30MB)
cols = ['station_name', 'datastream_id', 'station_id', 'service_name', 'layer_name', 'resolution',
        'longitude_wgs84', 'latitude_wgs84',
        'hour_utc', 'datetime_hamburg', 'date', 'year', 'month', 'hour_24',
        'bike_count_hourly',
        'extreme_flag_relative_p995', 'extreme_count_gt1000', 'extreme_count_gt2000',
        'readings_in_hour', 'readings_expected', 'coverage_pct', 'low_coverage', 'quality_flag']
df = df[cols].sort_values(['station_id', 'datetime_hamburg']).reset_index(drop=True)

df.to_csv(OUTPUT_PATH, index=False, compression='gzip')
print(f"Written: {OUTPUT_PATH}")
print(f"Final row count: {len(df):,}  Stations: {df['station_id'].nunique()}")

Written: output/hamburg_bike_counts_hourly_5min_clean_v2.csv.gz
Final row count: 3,227,633  Stations: 328


## Known limitations (left for the next step or later analysis -- intentionally not handled here)

- Of the 328 stations, only 241 have full coverage over this period (all 10,176 rows present);
  the other 87 had gaps or a delayed start/early stop. This notebook does not impute or reweight
  for that.
- `extreme_flag_relative_p995` and `extreme_count_gt1000` are two different extreme-value
  definitions that coexist -- pick whichever fits the analysis; using only one is not recommended.
